# 07 · Ensembles: Voting, Bagging, Boosting, Stacking (RQ4, RQ5)
Heterogeneous base learners (RF + XGB + LR + SVM). The weighted ensemble tunes its weights on the validation split. Also: per-attack-type recall (RQ8) and SHAP (RQ6).

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "ml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib, json
print("project root:", ROOT)

In [ ]:
from ml.models.ensembles import build_ensemble_models
from ml.training.evaluate import evaluate_model, per_class_recall
import time
s = joblib.load("ml/data/splits/dataset_splits.joblib")
X_train, y_train, X_val, y_val, X_test, y_test = s["X_train"], s["y_train"], s["X_val"], s["y_val"], s["X_test"], s["y_test"]
opt_rf = joblib.load("ml/artifacts/models/optimized_random_forest.joblib") if Path("ml/artifacts/models/optimized_random_forest.joblib").exists() else None
opt_xgb = joblib.load("ml/artifacts/models/optimized_xgboost.joblib") if Path("ml/artifacts/models/optimized_xgboost.joblib").exists() else None
rows, fitted = [], {}
for name, model in build_ensemble_models(random_state=42, optimized_rf=opt_rf, optimized_xgb=opt_xgb).items():
    t = time.perf_counter()
    model.fit(X_train, y_train, X_val=X_val, y_val=y_val) if name == "weighted_ensemble" else model.fit(X_train, y_train)
    tt = time.perf_counter() - t
    ev = evaluate_model(model, X_test, y_test, name, training_time=tt); fitted[name] = model
    rows.append({"ensemble": name, "f1": ev["f1"], "recall": ev["recall"], "fpr": ev["false_positive_rate"], "roc_auc": ev["roc_auc"], "train_s": round(tt, 1), "infer_ms_1k": ev["inference_time_ms_per_1k"]})
    print(f"{name:22s} F1 {ev['f1']:.4f} recall {ev['recall']:.4f} FPR {ev['false_positive_rate']:.4f}")
pd.DataFrame(rows).sort_values("f1", ascending=False)

In [ ]:
# Per-attack-type recall of the stacking ensemble
pd.DataFrame(per_class_recall(fitted["stacking"], X_test, y_test, s["y_test_labels"]))

In [ ]:
# SHAP on the tuned XGBoost (exact TreeExplainer)
from ml.explainability.shap_analysis import IDSExplainer
if opt_xgb is not None:
    exp = IDSExplainer(opt_xgb, feature_names=s["feature_names"], background_data=X_train[:200])
    imp = pd.DataFrame(exp.explain_global(X_test[:500], max_display=15))
    imp.set_index("feature")["importance"].iloc[::-1].plot.barh(figsize=(7, 5), color="#0072B2"); plt.xlabel("mean |SHAP|"); plt.tight_layout()
    imp